# 📊 WhiskerRegressionNet Training

This notebook trains a neural network to regress whisker positions from box plot statistics.

**Purpose**: Learn to predict whisker_low and whisker_high from Q1, Q3, IQR, median, and outlier statistics.

**Training data**: Generated synthetically using `generate_whisker_training_data.py` - ground truth from matplotlib boxplots.

## Setup

In [ ]:
# Install dependencies
!pip install torch tqdm matplotlib numpy -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from google.colab import files
import zipfile
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Upload Training Data

Upload the `whisker_training_data.json` generated by `generate_whisker_training_data.py`

In [ ]:
# Upload the training data
uploaded = files.upload()

# Load JSON
filename = list(uploaded.keys())[0]
if filename.endswith('.zip'):
    with zipfile.ZipFile(filename, 'r') as zf:
        zf.extractall('training_data')
    with open('training_data/whisker_training_data.json', 'r') as f:
        raw_data = json.load(f)
else:
    with open(filename, 'r') as f:
        raw_data = json.load(f)

print(f"Loaded {len(raw_data)} charts")
total_boxes = sum(chart['num_boxes'] for chart in raw_data)
print(f"Total boxes: {total_boxes}")

## Data Processing

Convert raw annotations to training format:
- **Input features**: Q1, Q3, IQR, median, median_confidence, outlier_count, outliers_below_q1, outliers_above_q3
- **Output targets**: whisker_low, whisker_high (or whisker ratios)

In [ ]:
class WhiskerDataset(torch.utils.data.Dataset):
    """
    Dataset for whisker regression.
    
    Features (input):
        - q1, q3, iqr, median (normalized by data range)
        - median_confidence
        - outlier_count, outliers_below_q1, outliers_above_q3 (counts)
    
    Targets (output):
        - whisker_low_ratio: (q1 - whisker_low) / iqr
        - whisker_high_ratio: (whisker_high - q3) / iqr
    """
    
    def __init__(self, raw_data, normalize=True):
        self.samples = []
        self.normalize = normalize
        
        for chart in raw_data:
            for box in chart['boxes']:
                # Skip boxes with zero IQR (degenerate)
                if box['iqr'] <= 0:
                    continue
                
                # Input features
                features = np.array([
                    box['q1'],
                    box['q3'],
                    box['iqr'],
                    box['median'],
                    box['median_confidence'],
                    box['features']['outlier_count'],
                    box['features']['outliers_below_q1'],
                    box['features']['outliers_above_q3'],
                ], dtype=np.float32)
                
                # Target: whisker ratios (relative to IQR)
                whisker_low_ratio = (box['q1'] - box['whisker_low']) / box['iqr']
                whisker_high_ratio = (box['whisker_high'] - box['q3']) / box['iqr']
                
                targets = np.array([
                    whisker_low_ratio,
                    whisker_high_ratio
                ], dtype=np.float32)
                
                # Also store absolute values for evaluation
                absolute = np.array([
                    box['whisker_low'],
                    box['whisker_high']
                ], dtype=np.float32)
                
                self.samples.append({
                    'features': features,
                    'targets': targets,
                    'absolute': absolute,
                    'box_info': box
                })
        
        # Compute normalization stats
        if normalize and self.samples:
            all_features = np.stack([s['features'] for s in self.samples])
            self.feature_mean = all_features.mean(axis=0)
            self.feature_std = all_features.std(axis=0) + 1e-8
        else:
            self.feature_mean = np.zeros(8)
            self.feature_std = np.ones(8)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        features = sample['features'].copy()
        
        if self.normalize:
            features = (features - self.feature_mean) / self.feature_std
        
        return {
            'features': torch.tensor(features, dtype=torch.float32),
            'targets': torch.tensor(sample['targets'], dtype=torch.float32),
            'absolute': torch.tensor(sample['absolute'], dtype=torch.float32)
        }

In [ ]:
# Create dataset
dataset = WhiskerDataset(raw_data, normalize=True)
print(f"Dataset size: {len(dataset)} boxes")

# Split: 80% train, 10% val, 10% test
n = len(dataset)
train_size = int(0.8 * n)
val_size = int(0.1 * n)
test_size = n - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# Create data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32)

## WhiskerRegressionNet Model

Simple feedforward network that predicts whisker ratios from box statistics.

In [ ]:
class WhiskerRegressionNet(nn.Module):
    """
    Neural network for whisker position regression.
    
    Architecture:
    - Input: 8 features (q1, q3, iqr, median, confidence, outlier stats)
    - 3 hidden layers with ReLU + Dropout
    - Output: 2 values (whisker_low_ratio, whisker_high_ratio)
    - Uncertainty: Optional aleatoric uncertainty estimation
    """
    
    def __init__(self, input_dim=8, hidden_dim=64, output_dim=2, with_uncertainty=True):
        super().__init__()
        self.with_uncertainty = with_uncertainty
        
        # Shared feature extractor
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU()
        )
        
        # Mean prediction head
        self.mean_head = nn.Linear(hidden_dim // 2, output_dim)
        
        # Uncertainty prediction head (log variance)
        if with_uncertainty:
            self.log_var_head = nn.Linear(hidden_dim // 2, output_dim)
    
    def forward(self, x):
        features = self.encoder(x)
        mean = self.mean_head(features)
        
        if self.with_uncertainty:
            log_var = self.log_var_head(features)
            return mean, log_var
        else:
            return mean, None
    
    def predict(self, x):
        """Inference mode: return mean + uncertainty."""
        self.eval()
        with torch.no_grad():
            mean, log_var = self.forward(x)
            if log_var is not None:
                std = torch.exp(0.5 * log_var)
                return {
                    'whisker_low_ratio': mean[:, 0],
                    'whisker_high_ratio': mean[:, 1],
                    'uncertainty_low': std[:, 0],
                    'uncertainty_high': std[:, 1]
                }
            else:
                return {
                    'whisker_low_ratio': mean[:, 0],
                    'whisker_high_ratio': mean[:, 1]
                }

model = WhiskerRegressionNet(input_dim=8, hidden_dim=64, with_uncertainty=True).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
def gaussian_nll_loss(mean, log_var, target):
    """
    Negative log-likelihood loss for Gaussian with learned variance.
    This enables uncertainty quantification.
    
    NLL = 0.5 * (log_var + (target - mean)^2 / exp(log_var))
    """
    precision = torch.exp(-log_var)
    return 0.5 * torch.mean(log_var + precision * (target - mean) ** 2)


def train_epoch(model, loader, optimizer, with_uncertainty=True):
    model.train()
    total_loss = 0
    
    for batch in loader:
        features = batch['features'].to(device)
        targets = batch['targets'].to(device)
        
        optimizer.zero_grad()
        mean, log_var = model(features)
        
        if with_uncertainty and log_var is not None:
            loss = gaussian_nll_loss(mean, log_var, targets)
        else:
            loss = F.mse_loss(mean, targets)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(features)
    
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, with_uncertainty=True):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    for batch in loader:
        features = batch['features'].to(device)
        targets = batch['targets'].to(device)
        
        mean, log_var = model(features)
        
        if with_uncertainty and log_var is not None:
            loss = gaussian_nll_loss(mean, log_var, targets)
        else:
            loss = F.mse_loss(mean, targets)
        
        total_loss += loss.item() * len(features)
        all_preds.append(mean.cpu())
        all_targets.append(targets.cpu())
    
    # Compute MAE
    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)
    mae = torch.abs(preds - targets).mean().item()
    
    return total_loss / len(loader.dataset), mae

In [ ]:
# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

NUM_EPOCHS = 100
best_val_mae = float('inf')
train_losses, val_losses = [], []
train_maes, val_maes = [], []

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_loss, val_mae = eval_epoch(model, val_loader)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_maes.append(val_mae)
    
    # Save best model
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), 'best_whisker_model.pt')
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val MAE: {val_mae:.4f}")

print(f"\nBest Val MAE: {best_val_mae:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses, label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (NLL)')
axes[0].legend()
axes[0].set_title('Loss Curves')

axes[1].plot(val_maes, label='Val MAE', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Absolute Error')
axes[1].legend()
axes[1].set_title('Validation MAE (Whisker Ratio)')

plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
# Load best model and evaluate on test set
model.load_state_dict(torch.load('best_whisker_model.pt'))
test_loss, test_mae = eval_epoch(model, test_loader)

print(f"Test Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  MAE (Whisker Ratio): {test_mae:.4f}")

In [ ]:
# Detailed evaluation: Convert predictions back to absolute whisker values
model.eval()
absolute_errors = []

with torch.no_grad():
    for batch in test_loader:
        features = batch['features'].to(device)
        absolute = batch['absolute']  # Ground truth whisker_low, whisker_high
        
        # Get predictions (ratios)
        mean, log_var = model(features)
        pred_ratios = mean.cpu()
        
        # Get original (unnormalized) box stats from dataset
        for i in range(len(features)):
            # Note: We need to recover Q1, Q3, IQR from features
            # Since features are normalized, we use absolute values directly
            gt_low, gt_high = absolute[i].numpy()
            pred_low_ratio, pred_high_ratio = pred_ratios[i].numpy()
            
            # For now, just compute ratio MAE (we need denormalization for absolute)
            absolute_errors.append({
                'gt_low': gt_low,
                'gt_high': gt_high,
                'pred_low_ratio': pred_low_ratio,
                'pred_high_ratio': pred_high_ratio
            })

print(f"\nTest Set Analysis ({len(absolute_errors)} samples):")
print(f"  Whisker Low Ratio MAE: {np.mean([abs(e['pred_low_ratio']) for e in absolute_errors]):.4f}")
print(f"  Whisker High Ratio MAE: {np.mean([abs(e['pred_high_ratio']) for e in absolute_errors]):.4f}")

## Export Model

In [ ]:
# Save model for deployment
export_dict = {
    'model_state_dict': model.state_dict(),
    'config': {
        'input_dim': 8,
        'hidden_dim': 64,
        'output_dim': 2,
        'with_uncertainty': True
    },
    'normalization': {
        'feature_mean': dataset.feature_mean.tolist(),
        'feature_std': dataset.feature_std.tolist()
    },
    'metrics': {
        'test_loss': test_loss,
        'test_mae': test_mae
    }
}

torch.save(export_dict, 'whisker_regression_net_final.pt')

# Download
files.download('whisker_regression_net_final.pt')
print("Model exported and downloaded!")

## Usage Example

Example of how to use the trained model for inference:

In [ ]:
def load_whisker_model(checkpoint_path):
    """Load trained WhiskerRegressionNet from checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    model = WhiskerRegressionNet(**checkpoint['config'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model, checkpoint['normalization']


def predict_whiskers(model, norm_stats, q1, q3, median, median_confidence=1.0, 
                     outlier_count=0, outliers_below_q1=0, outliers_above_q3=0):
    """
    Predict whisker positions from box statistics.
    
    Args:
        model: Trained WhiskerRegressionNet
        norm_stats: Normalization statistics from checkpoint
        q1, q3, median: Box plot quartiles
        median_confidence: Confidence in median detection (0-1)
        outlier_count: Number of outliers
        outliers_below_q1: Outliers below Q1
        outliers_above_q3: Outliers above Q3
    
    Returns:
        dict with whisker_low, whisker_high, and uncertainties
    """
    iqr = q3 - q1
    
    # Build feature vector
    features = np.array([
        q1, q3, iqr, median, median_confidence,
        outlier_count, outliers_below_q1, outliers_above_q3
    ], dtype=np.float32)
    
    # Normalize
    mean = np.array(norm_stats['feature_mean'])
    std = np.array(norm_stats['feature_std'])
    features_norm = (features - mean) / std
    
    # Predict
    x = torch.tensor(features_norm).unsqueeze(0)
    with torch.no_grad():
        pred_mean, pred_log_var = model(x)
    
    # Convert ratios to absolute values
    low_ratio = pred_mean[0, 0].item()
    high_ratio = pred_mean[0, 1].item()
    
    whisker_low = q1 - low_ratio * iqr
    whisker_high = q3 + high_ratio * iqr
    
    # Uncertainties
    if pred_log_var is not None:
        std_low = torch.exp(0.5 * pred_log_var[0, 0]).item()
        std_high = torch.exp(0.5 * pred_log_var[0, 1]).item()
    else:
        std_low = std_high = 0.0
    
    return {
        'whisker_low': whisker_low,
        'whisker_high': whisker_high,
        'uncertainty_low': std_low * iqr,  # Convert to absolute uncertainty
        'uncertainty_high': std_high * iqr
    }


# Example usage
model, norm_stats = load_whisker_model('whisker_regression_net_final.pt')

result = predict_whiskers(
    model, norm_stats,
    q1=25.0, q3=75.0, median=50.0,
    median_confidence=0.9,
    outlier_count=2, outliers_below_q1=1, outliers_above_q3=1
)

print(f"Predicted Whiskers:")
print(f"  Low: {result['whisker_low']:.2f} ± {result['uncertainty_low']:.2f}")
print(f"  High: {result['whisker_high']:.2f} ± {result['uncertainty_high']:.2f}")